# Tree GeoJSON Conversion

This notebook converts Roboflow tree detections from Vienna orthofoto image pixel coordinates into longitude/latitude points.

The output is saved as `outputs/detected_trees.geojson`.

In [ ]:
# Import the libraries we need.
# json lets us read and write prediction/GeoJSON files.
# sys lets us add the project folder to Python's import path.
# Path helps us build file paths that work on different operating systems.
import json
import sys
from pathlib import Path

In [ ]:
# Find the project root folder.
# If this notebook is opened from the notebooks/ folder, the project root is one level up.
# If it is opened from the project root, the current folder is already the project root.
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path.
# This lets the notebook import code from backend/app.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
# Import the reusable GeoJSON builder from the backend.
# Reload the module so Jupyter picks up recent edits without requiring a kernel restart.
# It converts Roboflow pixel coordinates into lon/lat GeoJSON points.
import importlib
import backend.app.simulation.vegetation_builder as vegetation_builder

vegetation_builder = importlib.reload(vegetation_builder)
detections_to_tree_geojson = vegetation_builder.detections_to_tree_geojson

In [ ]:
# Set the file paths used by this notebook.
# The predictions file is created by notebooks/02_roboflow_tree_detection.ipynb.
image_path = project_root / "data" / "vienna_orthofoto_test.png"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"
predictions_path = project_root / "outputs" / "roboflow_predictions.json"
geojson_path = project_root / "outputs" / "detected_trees.geojson"

# Stop early with clear messages if the previous steps have not been run yet.
if not image_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto image: {image_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

if not predictions_path.exists():
    raise FileNotFoundError(
        f"Missing predictions file: {predictions_path}. Run notebooks/02_roboflow_tree_detection.ipynb first."
    )

In [ ]:
# Load the Vienna metadata and Roboflow predictions from notebook 02.
# The file is expected to contain a list of prediction dictionaries.
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
predictions_data = json.loads(predictions_path.read_text(encoding="utf-8"))

# Notebook 02 now saves predictions with input metadata.
# This prevents accidentally converting old detections against a new orthofoto bbox.
if not isinstance(predictions_data, dict) or "input_metadata" not in predictions_data:
    raise ValueError(
        "Roboflow predictions are missing input_metadata. Rerun notebook 02 after changing notebook 01b coordinates."
    )

prediction_metadata = predictions_data["input_metadata"]
if prediction_metadata.get("center") != metadata.get("center") or prediction_metadata.get("bbox") != metadata.get("bbox"):
    raise ValueError(
        "Roboflow predictions metadata does not match the current Vienna orthofoto metadata. "
        "Rerun notebook 02 before running this conversion."
    )

predictions = predictions_data.get("predictions", [])
tree_predictions = [
    prediction
    for prediction in predictions
    if str(prediction.get("class", "")).strip().lower() == "tree"
]

print(f"Loaded predictions: {len(predictions)}")
print(f"Tree-class predictions: {len(tree_predictions)}")

In [ ]:
# Load the Vienna orthofoto image and metadata so we know the exact pixel-to-coordinate mapping.
# Roboflow prediction x/y coordinates are based on this image size.
from PIL import Image

orthofoto_image = Image.open(image_path)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
bbox = metadata["bbox"]
image_width, image_height = orthofoto_image.size
metadata_width = metadata["image_width"]
metadata_height = metadata["image_height"]

if (image_width, image_height) != (metadata_width, metadata_height):
    raise ValueError(
        f"Image size {image_width}x{image_height} does not match metadata {metadata_width}x{metadata_height}."
    )

print(f"Vienna orthofoto size: {image_width} x {image_height} pixels")
print(f"Default Vienna center: lat={metadata['center']['lat']}, lon={metadata['center']['lon']}")
print(f"CRS: {metadata['crs']}")

In [ ]:
# These settings come from the Vienna orthofoto metadata created by notebook 01b.
center_lat = metadata["center"]["lat"]
center_lon = metadata["center"]["lon"]
zoom = metadata["zoom"]
image_bbox = (bbox["min_x"], bbox["min_y"], bbox["max_x"], bbox["max_y"])

# Only include detections at or above this confidence score.
confidence_threshold = 0.35

In [ ]:
# Convert Roboflow detections into GeoJSON tree points.
# Each output feature is a Point with [longitude, latitude] coordinates.
# The builder also estimates canopy_radius_m from the Roboflow bbox size.
tree_geojson = detections_to_tree_geojson(
    predictions=tree_predictions,
    center_lon=center_lon,
    center_lat=center_lat,
    zoom=zoom,
    width=image_width,
    height=image_height,
    confidence_threshold=confidence_threshold,
    image_bbox=image_bbox,
    allowed_classes={"tree"},
)

tree_geojson["metadata"] = {
    "source_image": str(image_path),
    "imagery_source": metadata["source"],
    "center": metadata["center"],
    "bbox": metadata["bbox"],
    "bbox_lonlat": metadata["bbox_lonlat"],
    "crs": metadata["crs"],
    "image_width": image_width,
    "image_height": image_height,
}

# Save the GeoJSON file so later steps can use it for UTCI simulation and routing.
geojson_path.parent.mkdir(parents=True, exist_ok=True)
geojson_path.write_text(json.dumps(tree_geojson, indent=2), encoding="utf-8")

canopy_radii = [
    feature["properties"].get("canopy_radius_m")
    for feature in tree_geojson["features"]
    if feature["properties"].get("canopy_radius_m") is not None
]

print(f"Number of detected trees: {len(tree_geojson['features'])}")
if canopy_radii:
    print(f"Average estimated canopy radius: {sum(canopy_radii) / len(canopy_radii):.2f} m")
    print(f"Max estimated canopy radius: {max(canopy_radii):.2f} m")
print("First 5 GeoJSON features:")
print(json.dumps(tree_geojson["features"][:5], indent=2))
print(f"Output file path: {geojson_path}")

In [ ]:
# Display the Vienna orthofoto and plot the detected tree points on top.
# We use the original Roboflow pixel x/y values for the overlay, because they line up with the image.
import matplotlib.pyplot as plt

filtered_predictions = [
    prediction
    for prediction in tree_predictions
    if float(prediction.get("confidence", 0)) >= confidence_threshold
]

tree_x = [float(prediction["x"]) for prediction in filtered_predictions]
tree_y = [float(prediction["y"]) for prediction in filtered_predictions]

plt.figure(figsize=(8, 8))
plt.imshow(orthofoto_image)
tree_sizes = [
    max(20, min(float(prediction.get("width", 10)) * float(prediction.get("height", 10)) / 20, 220))
    for prediction in filtered_predictions
]

plt.scatter(tree_x, tree_y, s=tree_sizes, c="lime", edgecolors="black", linewidths=0.5, alpha=0.75)
plt.axis("off")
plt.title("Detected Tree Points on Vienna Orthofoto")
plt.show()